In [ ]:
from pcntoolkit import (
    HBR,
    LaPlace,
    NormativeModel,
    load_fcon1000,
)

In [ ]:
fcon = load_fcon1000()
fcon = fcon.sel({"response_vars": ["EstimatedTotalIntraCranialVol"]})
fcon_transfer, fcon_train = fcon.batch_effects_split(
    {"site": ["AnnArbor_a", "Beijing_Zang"]}, names=["train", "transfer"]
)
train, test = fcon_train.train_test_split((0.8, 0.2), split_names=("train", "test"))
transfer_train, transfer_test = fcon_transfer.train_test_split((0.8, 0.2), split_names=("train", "test"))

In [ ]:
# inference_method = MCMC(
#     draws=100, tune=50, cores=4, chains=4, init="auto", nuts_sampler="nutpie"
# )

inference_method = LaPlace(draws = 4000)

model = NormativeModel(
    HBR(inference_method=inference_method),
    save_dir="/Users/stijn.deboer/Projects/PCNtoolkit/examples/saves",
)
model.fit_predict(train, test)

In [ ]:
del model

In [ ]:
model = NormativeModel.load("/Users/stijn.deboer/Projects/PCNtoolkit/examples/saves")


In [ ]:
model.predict(test)

In [ ]:
model.transfer_predict(transfer_train, transfer_test)

In [ ]:
extended_model = model.extend_predict(transfer_train, transfer_test)

In [ ]:
extended_model.predict(fcon)